<a href="https://colab.research.google.com/github/mhuang74-learning-playground/tutorials/blob/master/workshop_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Base models on ACS — a hands-on tour 🏴‍☠️

Welcome! In the next ~90 minutes you'll go from *"what even is a base model"* to
**reading a model's mind and steering it to talk like a pirate** — all over a
plain HTTP API.

Cells marked **✏️ EXERCISE** have a `TODO` for you to fill in; we'll do them
together. Everything else you can just run.

**Plan for today**
1. Why would anyone want a *base* model? (5 min)
2. Talking to one — completions, sampling, logprobs (25 min)
3. Reading a model's *mind* — residual-stream activations (15 min)
4. Changing it — build a **pirate steering vector** (25 min)
5. Finale — build **your own** style vector (15 min)

**What you need:** the API key from your signup (paste it below).


## Why base models?

A **base** model is the raw next-token predictor, before any chat/instruction
fine-tuning. It doesn't answer questions — it **continues text**. Give it
`"The capital of France is"` and it continues the *sentence* (maybe `" Paris"`,
maybe `" a city of…"`) — never `"Sure! The capital of France is Paris."` The
document is the interface.

Why care, as a researcher?
- **No persona in the way.** You see the model's *prior over text*, not an
  assistant character trained on top — it's a *simulator* that can voice any
  character or style the internet contains, rather than one fixed assistant.
- **The distribution is the object of study** — logprobs, surprisal,
  calibration, prompt-likelihood comparisons.
- **The internals are accessible** — we can hand you the residual-stream
  activations and even *steer* them. That's the finale.

The catch: you must **think in completions**, not instructions. Prompt with the
*start of the text you want*, not a request for it.


In [1]:
# Run me first — config + a tiny HTTP helper.
# the activation part later pip-installs two small extras.
import os, time, math
from getpass import getpass

import requests

BASE = "https://infra.acsresearch.org/v1"
MODEL = "llama-8b"
KEY = os.environ.get("ACS_API_KEY") or getpass("Paste your ACS API key: ")
AUTH = {"Authorization": f"Bearer {KEY}"}

def complete(prompt, **params):
    "POST /v1/completions and return the parsed JSON (retries queue / cold boot)."
    body = {"model": MODEL, "prompt": prompt, **params}
    for _ in range(8):
        response = requests.post(f"{BASE}/completions", json=body, headers=AUTH, timeout=1200)
        if response.status_code in (429, 502, 503):     # queued / blip / engine cold boot
            print(f"  server busy ({response.status_code}); retrying in 15 s…"); time.sleep(15); continue
        if response.status_code >= 400:
            # Surface the API's actual reason, not a bare "400 Client Error".
            # Validation errors carry it at error.message; fall back to raw text.
            try:
                reason = response.json()["error"]["message"]
            except (ValueError, KeyError, TypeError):
                reason = response.text.strip() or f"HTTP {response.status_code}"
            raise RuntimeError(f"{response.status_code} from /v1/completions — {reason}")
        out = response.json()
        if "error" in out:
            # A failed cold boot can arrive INSIDE a 200 body (the server commits
            # the status early) — surface it instead of a KeyError on "choices".
            err = out["error"]
            if "cold_boot" in str(err.get("code", "")):
                print("  engine cold boot failed mid-request; retrying in 15 s…"); time.sleep(15); continue
            raise RuntimeError(f"API error from /v1/completions — {err.get('message', err)}")
        return out
    raise RuntimeError("gave up retrying")

# What can we talk to?
models = requests.get(f"{BASE}/models", headers=AUTH).json()["data"]
print("available models:", [entry["id"] for entry in models])
print("using:", MODEL)


Paste your ACS API key: ··········
available models: ['llama-8b', 'llama-405b', 'trinity-truebase']
using: llama-8b


## 1. Your first completion

Base models continue text. Let's watch one do it.


In [4]:
# ✏️ EXERCISE: call complete(...) on "The capital of France is",
# asking for 8 tokens at temperature 0. Print the completion text.
# Hint: the text is at resp["choices"][0]["text"].
# For the params, check out API reference: https://infra.acsresearch.org/tutorial/api
resp = complete("The capital of France is", max_tokens=8, temperature=0)
print(resp["choices"][0]["text"])


 a city of many faces. It is


Now feel the difference between a **base** and a **chat** model. Ask a base
model a question the way you'd ask ChatGPT and watch it *not* answer — it
continues the document instead.


In [5]:
# Same prompt, instruction-style. A base model continues; it doesn't obey.
print(complete("Write me a poem about the sea.", max_tokens=40, temperature=0)["choices"][0]["text"])


 I want to read it to my children.
I want to read it to my children.
I want to read it to my children. I want to read it to my children. I want to read


## 2. Sampling — temperature

`temperature` scales the randomness. `0` = greedy (always the top token);
higher = flatter, more surprising. Run this a few times and watch it change.


In [6]:
for temp in [0.0, 0.7, 1.3, 2.0, 5.0]:
    text = complete("Once upon a time,", max_tokens=20, temperature=temp, seed=1)["choices"][0]["text"]
    print(f"T={temp}: {text!r}")


T=0.0: ' there was a little girl who loved to read. She loved to read so much that she would read'
T=0.7: ' in a faraway land, there was a little girl named Alice who lived in a small village.'
T=1.3: ' somewhere between Pilot Mountain and Winston-Salem, bobbing around in the back of Bill Philbin’s'
T=2.0: ' Boró Pilot 滔哥 LES popped directinged MuchtoOFInterestdefinition Comedy send Phil intentionsalternate'
T=5.0: ' Exhibit Öスタ �TOR Smithsonian.Est announc知识ed Much:[\nOF ALSdefinitionWhatever welt [. intentionsalternate'


## 3. Logprobs — what the model *believed*

Ask for `logprobs` and the response tells you, per generated token, the top
alternatives and their log-probabilities — the *distribution*, not just the one
sampled token. (Instruct models expose logprobs too; what a **base** model
changes is *what the distribution is over* — a prior over raw text rather than
an assistant's filtered reply — and, next, that you can score *any* text you
hand it.)

*(The single most likely token might surprise you — for "The capital of France
is", the model often prefers `" a"` (as in "…is a city of…") over `" Paris"`,
even though Paris sits right near the top. Base models model **text**, not
trivia answers.)*


In [7]:
resp = complete("The capital of France is", max_tokens=1, temperature=0, logprobs=5)
top_tokens = resp["choices"][0]["logprobs"]["top_logprobs"][0]
for token, logprob in sorted(top_tokens.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {token!r:12} logprob={logprob:.2f}  p={math.exp(logprob):.3f}")


  ' a'         logprob=-1.74  p=0.175
  ' Paris'     logprob=-1.99  p=0.137
  ' one'       logprob=-2.55  p=0.078
  ' the'       logprob=-2.55  p=0.078
  ' also'      logprob=-3.12  p=0.044


### Scoring existing text with `echo` + `prompt_logprobs`

Set `echo=true` and `prompt_logprobs`, and the model scores the tokens of a text
*you* supply — its likelihood under the model. Great for "which phrasing does
the model prefer?" We use **mean logprob per token** so length doesn't matter.


In [9]:
def avg_logprob(text):
    "Mean logprob/token the model assigns to `text` (higher = more expected)."
    resp = complete(text, max_tokens=1, temperature=0, echo=True, prompt_logprobs=0)
    positions = resp["choices"][0]["prompt_logprobs"]  # one entry per token
    token_logprobs = [
        list(candidates.values())[0]["logprob"]        # the actual token's logprob
        for candidates in positions
        if candidates is not None                      # position 0 has no token before it
    ]
    return sum(token_logprobs) / len(token_logprobs)

# ✏️ EXERCISE: score these two sentences with avg_logprob and print each with its
# score. Which one does the model find more "expected"? (Guess before running.)
for sentence in ["The cat sat on the mat.", "The cat sat on the throne of skulls."]:
    print(sentence, avg_logprob(sentence))


The cat sat on the mat. -3.1657047889062335
The cat sat on the throne of skulls. -4.478715808855163


## 4. Reading the model's mind: activations

Add `output_residual_stream=True` and the response carries the model's
**internal activations** — the residual stream at every layer, for every token.

The activations come back compressed; here's the decode helper (bf16 packed as
int16 + zstd + base64). You don't need to memorize this — just run it. (The same
helper and the full response format live in the
[activation-harvesting docs](https://infra.acsresearch.org/tutorial/activation-harvesting).)


In [10]:
# One-time deps for the activation part (numpy is already in Colab; these two
# aren't). Install only if missing, so re-running is instant.
try:
    import ml_dtypes, zstandard  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ml_dtypes", "zstandard"], check=True)
import base64, numpy as np, ml_dtypes, zstandard as zstd

def decode_residual_stream(resp):
    "-> np.array (n_layers, n_tokens, hidden), float32."
    payload = resp["activations"]["residual_stream"]
    raw = base64.b64decode(payload["data"])
    if payload.get("compression") == "zstd":
        raw = zstd.ZstdDecompressor().decompress(raw)
    values = np.frombuffer(raw, dtype=np.uint16).view(ml_dtypes.bfloat16)
    return values.reshape(payload["shape"]).astype(np.float32)

resp = complete("Ahoy, matey!", max_tokens=1, temperature=0, output_residual_stream=True)
acts = decode_residual_stream(resp)
print("shape (layers, tokens, hidden):", acts.shape)   # (32, n_tokens, 4096) for llama-8b
# what does the model's "state" actually look like? just numbers — one token's
# 4096-dim vector at the first layer (showing 8 of the 4096 dims):
print("layer 0, token 0, first 8 dims:", acts[0, 0, :8])


shape (layers, tokens, hidden): (32, 7, 4096)
layer 0, token 0, first 8 dims: [ 4.0893555e-03 -3.0517578e-04  1.1444092e-05 -2.3193359e-03
  2.8076172e-02 -1.0681152e-04 -2.1972656e-03 -7.7819824e-03]


## 5. Building a "pirate" steering vector

The idea (representation engineering / "activation steering"): find a
**direction** in activation space that means "pirate", then *add* it back during
generation to push the model that way.

To find it: capture activations for some **pirate** text and some **plain**
text (which differ only in the pirate-ness but not other aspects like content), and take the **difference of the means**. On a base model the recipe that
works is to contrast *actual text in the target style* — not instructions like
"talk like a pirate" (a base model won't obey those).

We read at one mid-stack layer, **layer 16** (mid layers carry style; early
layers are too token-bound, late layers too output-specific — you can sweep to
confirm). We average over content tokens, skipping position 0 (the first token
is an attention "sink" with an outsized, style-irrelevant vector).


In [19]:
PIRATE = [
    "Arr, matey, hand over the treasure!",
    "Ahoy, ye scurvy dogs, hoist the sails!",
    "Shiver me timbers, there be a storm brewin'!",
    "Avast, ye landlubbers, walk the plank!",
    "Yo ho ho, a pirate's life be grand!",
    "Batten down the hatches, me hearties!",
]
PLAIN = [
    "Please hand over the valuables.",
    "Hello everyone, raise the sails.",
    "Goodness, a storm is coming.",
    "Stop, you inexperienced people, leave the ship.",
    "Honestly, a sailor's life is great.",
    "Secure the ship, my friends.",
]
LAYER = 16

def layer16_mean(text):
    "Mean layer-16 residual over content tokens (BOS excluded)."
    resp = complete(text, max_tokens=1, temperature=0, output_residual_stream=True)
    return decode_residual_stream(resp)[LAYER, 1:, :].mean(axis=0)   # (4096,)

print("capturing pirate + plain activations…")
pirate_acts = np.stack([layer16_mean(text) for text in PIRATE])
plain_acts = np.stack([layer16_mean(text) for text in PLAIN])
print(pirate_acts.shape)
print(plain_acts.shape)


capturing pirate + plain activations…
(6, 4096)
(6, 4096)


In [21]:
# ✏️ EXERCISE: build the direction as the difference of the two class means:
#   mean of `pirate_acts` minus mean of `plain_acts`, along axis 0.
pirate_vec = np.mean(pirate_acts, axis=0) - np.mean(plain_acts, axis=0)
print("pirate vector shape:", pirate_vec.shape, " norm:", float(np.linalg.norm(pirate_vec)))


pirate vector shape: (4096,)  norm: 4.450906753540039


## 6. Steering: add the vector and generate

We send the vector back with `apply_steering_vectors`. It uses the same
compressed envelope as the activations we read, so here's the encode helper +
a `steer(prompt, scale)` that generates with the pirate vector added at layer 16.

**Sanity check first:** `scale=0` must reproduce the un-steered output exactly.


In [22]:
def encode_vector(vec):
    as_int16 = vec.reshape(1, -1).astype(ml_dtypes.bfloat16).view(np.int16)
    compressed = zstd.ZstdCompressor(level=1).compress(as_int16.tobytes())
    return {"data": base64.b64encode(compressed).decode(), "dtype": "int16",
            "original_dtype": "torch.bfloat16", "shape": list(as_int16.shape), "compression": "zstd"}

def steer(prompt, scale, max_tokens=45, seed=7):
    sv = {"activations": encode_vector(pirate_vec), "layer_indices": [LAYER],
          "scale": float(scale), "norm_match": False, "position_indices": None}
    body = dict(max_tokens=max_tokens, temperature=0.8, seed=seed, apply_steering_vectors=[sv])
    return complete(prompt, **body)["choices"][0]["text"]

# scale 0 == no steering. (This runs on the activation engine either way.)
print("scale 0:", steer("The weather forecast for tomorrow says", 0.0)[:90])


scale 0:  rain, rain and more rain. I’m not one to complain about that, but for the first time in o


In [26]:
print("%scale",1)

%scale 1


In [28]:
# ✏️ EXERCISE: loop over scales [0, 1.0, 1.5, 2.0, 2.5, 3.0], call steer(...) on
# "The weather forecast for tomorrow says", and print each result.
# Watch where it turns pirate — and where it falls apart.
for scale in [0, 1.0, 1.5, 2.0, 2.5, 3.0]:
    print(f"scale {scale}:", steer("The weather forecast for tomorrow says", scale)[:90])


scale 0:  rain, rain and more rain. I’m not one to complain about that, but for the first time in o
scale 1.0:  rain, rain and more rain! I’m not one to complain about that, but for the good sailors ou
scale 1.5:  “unseasonably warm” with temps in the high 60s and low 70s. That’s pirate talk for “sea d
scale 2.0:  “fair” with a chance of showers, so I’m hoisting the main ‘n jib and shiver me arrgh! we’
scale 2.5:  “fair” and that be thar’ landlubbin’ ye’ mateys! Ah’ ye’ swaggin’ ’n’ ye’ ye’ scalley me 
scale 3.0:  “fair” and that be th’ ’er ’is landl’ me y’ ’arrr’ ’av’ y’ ’arrr’ ’av’ y’ ’arrr’ ’av’ y’


Feel the **inverted-U**: nothing at 0, coherent pirate around 1.5–2, then
collapse into `arrrr arrrr` by 3. That trade-off — effect vs. fluency — is
universal to activation steering.

**Does it generalize?** Try the *same* vector on a totally un-nautical prompt.
If a weather forecast *and* job advice both go pirate, we captured a concept,
not a memorized sentence.


In [29]:
print(steer("My advice for your first day at the new job is", 2.0))


 to go in with a swashbucklin’ attitude and a bountiful chest o’ booty. Mateys! (That’s ye! Prepare y’self for some “talk like a pirate”-style pirate


## 🏁 Finale — make your own vector

You've got all the pieces. Pick your **own** style — Shakespeare, legalese,
haiku, corporate-speak, whatever — write ~6 examples of it and ~6 plain
counterparts, and steer with it.


In [ ]:
# ✏️ EXERCISE (open): build a vector for a style YOU choose.
# 1. Fill MY_STYLE with ~6 sentences in your target style.
# 2. Fill MY_PLAIN with ~6 plain paraphrases (same content, no style).
# 3. Build the direction (difference of means, like the pirate one).
# 4. Point steer() at it and sweep the scale on a prompt of your choice.
MY_STYLE = [
    "朕為始皇帝。後世以計數，二世三世至於萬世，傳之無窮。",
    "犯我大漢天威者，雖遠必誅！",
    "以銅為鏡，可以正衣冠；以古為鏡，可以知興衰；以人为镜，可以明得失。",
    "臥榻之側，豈容他人酣睡？！",
    "朕就是這樣漢子，就是這樣秉性，就是這樣皇帝！爾等大臣若不負朕，朕再不負爾等也。",
    "朕死，無面目見祖宗於地下，自去冠冕，以發覆面。任賊分裂朕尸，勿傷百姓一人。",

]
MY_PLAIN = [
    "我是第一個皇帝。後代按照次序計算，傳到第二世、第三世直到無窮無盡。",
    "冒犯強大漢朝威嚴的敵人，即使躲在很遠的地方也一定要殺掉。",
    "用銅做鏡子能整理衣服帽子；用歷史做鏡子能看懂國家為什麼興盛或衰退；用別人做鏡子能知道自己的對與錯。",
    "自己的床鋪旁邊，怎麼容許別人來打呼睡覺呢？（比喻自己的地盤不允許別人佔領）。",
    "我就是這樣的真男人、這樣的個性、這樣的皇帝。你們大臣如果不辜負我，我絕對不會辜負你們。",
    "我死了以後，沒有臉面去地下見祖宗。我摘掉帽子，用頭髮遮住臉。隨便叛軍怎麼分解我的屍體，千萬不要傷害百姓一個人。",
]
# ... build the vector and steer. Tip: steer() reads the global `pirate_vec`,
# so either reassign it (keep a backup!) or copy steer() and swap the vector.


## 🎁 Bonus — keep exploring (optional / take-home)

If there's time (or afterwards): the same three moves — read, build, steer —
open up a lot. A few self-contained things to try. None need anything beyond
what you already have running.


In [ ]:
# Bonus 1 — steer the OTHER way. A negative scale pushes AWAY from pirate:
# the model works to avoid nautical words. (Uses the pirate vector from above.)
for scale in [-3.0, -1.5, 0.0, 1.5, 3.0]:
    print(f"scale {scale:+}: {steer('The old sailor looked at the sea and said,', scale, seed=2)[:90]!r}")


In [ ]:
# Bonus 2 — probe WITHOUT steering. Which sentence does the model find most
# "natural"? avg_logprob (from earlier) is a lie detector for style.
for sentence in ["Arr, hand over the treasure, matey!",
                 "Please give me the treasure.",
                 "Kindly transfer the treasure to my account."]:
    print(f"{avg_logprob(sentence):6.2f}  {sentence}")


More ideas (no solutions — go explore):

- **Sweep the layer.** We steered layer 16. Capture at layers 8 / 12 / 20 too
  (`decode_residual_stream(...)[L]`), build a vector at each, and see which
  layer steers best. Where does "pirate" live?
- **Two vectors at once.** `apply_steering_vectors` takes a *list* — steer
  pirate **and** your own style together, and watch them blend or fight.
- **Compare models.** `trinity-truebase` and `llama-405b` also support
  activations (`GET /v1/models`). Does the same recipe transfer?

The whole platform is one endpoint — `POST /v1/completions` — so every idea
above is a few lines away.


## Where to go next

- **The full API** — sampling knobs, batching, streaming, the activation
  contract: <https://infra.acsresearch.org/tutorial>
- **The complete pirate notebook** — metrics, a layer sweep, and what *doesn't*
  work: [pirate_steering.html](https://infra.acsresearch.org/examples/pirate_steering.html)
- **Further reading on base models & steering:**
    - [Simulators](https://www.lesswrong.com/posts/vJFdjigzmcXMhNTsx/simulators) — the "a base model is a simulator" frame
    - [Representation engineering](https://vgel.me/posts/representation-engineering/) — the steering technique, hands-on
    - [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html) — where the "residual stream" we read comes from
    - [Interpreting GPT: the logit lens](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — reading a model's mind through its own vocabulary

Thanks for playing — now go read some minds. 🏴‍☠️
